In [18]:
import os
import csv
from dotenv import load_dotenv
from typing import TypedDict, List

# LangChain imports
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

# LangGraph imports
from langgraph.graph import StateGraph, START, END

# Load environment variables
load_dotenv()

# Verify API key is loaded
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables. Please set it in your .env file.")

print("All libraries imported successfully!")
print("OpenAI API key loaded")

All libraries imported successfully!
OpenAI API key loaded


In [25]:
# Load the Paul Graham essays using Python's csv module
data_path = "pual_graham_essays.csv"

essays = []
with open(data_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        essays.append(row)

# Display basic information
print(f"Loaded {len(essays)} essays")
print(f"\nColumns: {list(essays[0].keys())}")
print(f"\nFirst few rows:")
for i, essay in enumerate(essays[:3]):
    print(f"\n{i+1}. {essay['title']} ({essay['date']})")

Loaded 3 essays

Columns: ['\ufeff"id"', 'title', 'date', 'text']

First few rows:

1. The Age of the Essay (2026-05-23T04:15:53.062769Z)

2. A Plan for Spam (2026-05-23T04:15:53.062769Z)

3. The Trouble with the Segway (2026-05-23T04:15:53.062769Z)


In [26]:
# Let's look at one essay to understand the content
sample_essay = essays[0]
print(f"Title: {sample_essay['title']}")
print(f"Date: {sample_essay['date']}")
print(f"\nFirst 500 characters of text:")
print(sample_essay['text'][:500] + "...")

Title: The Age of the Essay
Date: 2026-05-23T04:15:53.062769Z

First 500 characters of text:
Remember the essays you had to write in high school? Topic sentence, introductory paragraph, supporting paragraphs, conclusion. The conclusion being, say, that Ahab in _Moby Dick_ was a Christ-like figure....


In [28]:
# Convert essays to LangChain Document objects
documents = []

for essay in essays:
    doc = Document(
        page_content=essay['text'],
        metadata={
            'title': essay['title'],
            'date': essay['date'],
            'essay_id': essay['\ufeff"id"']
        }
    )
    documents.append(doc)

print(f"Created {len(documents)} documents")
print(f"\nExample document:")
print(f"Content length: {len(documents[0].page_content)} characters")
print(f"Metadata: {documents[0].metadata}")

Created 3 documents

Example document:
Content length: 205 characters
Metadata: {'title': 'The Age of the Essay', 'date': '2026-05-23T04:15:53.062769Z', 'essay_id': '1'}


In [29]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]  # Try to split at paragraph, then line, then word
)

# Split all documents into chunks
chunks = text_splitter.split_documents(documents)

print(f"Original documents: {len(documents)}")
print(f"Total chunks after splitting: {len(chunks)}")
print(f"Average chunks per document: {len(chunks) / len(documents):.1f}")
print(f"\nExample chunk:")
print(f"Content: {chunks[0].page_content[:300]}...")
print(f"Metadata: {chunks[0].metadata}")

Original documents: 3
Total chunks after splitting: 3
Average chunks per document: 1.0

Example chunk:
Content: Remember the essays you had to write in high school? Topic sentence, introductory paragraph, supporting paragraphs, conclusion. The conclusion being, say, that Ahab in _Moby Dick_ was a Christ-like figure....
Metadata: {'title': 'The Age of the Essay', 'date': '2026-05-23T04:15:53.062769Z', 'essay_id': '1'}


In [30]:
# Initialize OpenAI embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print("Embeddings model initialized")
print("Model: text-embedding-3-small")

# Optional: Test the embeddings with a sample text
sample_text = "What is a startup?"
sample_embedding = embeddings.embed_query(sample_text)
print(f"\nSample embedding dimensions: {len(sample_embedding)}")
print(f"First 5 values: {sample_embedding[:5]}")

Embeddings model initialized
Model: text-embedding-3-small


PermissionDeniedError: Error code: 403 - {'error': {'message': "key not allowed to access model. This key can only access models=['gemini-2.5-flash', 'gemini-2.5-pro', 'gemini-3.5-flash', 'gemini-3.1-flash-lite']. Tried to access text-embedding-3-small", 'type': 'key_model_access_denied', 'param': 'model', 'code': '403'}}

In [31]:
# Create vector store using LangChain's Chroma
print("Creating Chroma vector store... (this may take a minute)")

vectorstore = Chroma(
    collection_name="paul_graham_essays",
    embedding_function=embeddings,
    # persist_directory="./chroma_db"  # Uncomment to persist to disk
)

# Add all document chunks to the vector store
# This will automatically generate embeddings and store them
print("Adding documents and generating embeddings...")
vectorstore.add_documents(documents=chunks)

print(f"\nVector store created successfully!")
print(f"Total chunks indexed: {len(chunks)}")
print("\nThe indexing phase is complete. Your knowledge base is ready for queries!")

Creating Chroma vector store... (this may take a minute)
Adding documents and generating embeddings...


PermissionDeniedError: Error code: 403 - {'error': {'message': "key not allowed to access model. This key can only access models=['gemini-2.5-flash', 'gemini-2.5-pro', 'gemini-3.5-flash', 'gemini-3.1-flash-lite']. Tried to access text-embedding-3-small", 'type': 'key_model_access_denied', 'param': 'model', 'code': '403'}}